In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import SGDClassifier
import numpy as np
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models import infer_signature
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

/home/alina/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocessing_data_frame(frame):
    df = frame.copy()
    cat_columns = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
    num_columns = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']
    
    # Анализ и очистка данных
    
    # здравый смысл
    question_engine = df[(df["RestingBP"] < 30) | (df["RestingBP"] > 250)] #уровень артериального давления у человека в сознании не м.б < 50  и > 250
    df = df.drop(question_engine.index)
    
    # здравый смысл
    question_engine = df[(df["Cholesterol"] == 0) | (df["Cholesterol"] > 600)] # то же для холестирина
    df = df.drop(question_engine.index) 
    
    # здравый смысл
    question_price = df[(df["Age"] <= 0) | (df["Age"] > 120)]
    df = df.drop(question_price.index)

    # здравый смысл
    question_price = df[~df['Sex'].isin(['M', 'F'])]
    df = df.drop(question_price.index)  
    
    df = df.reset_index(drop=True)  # обновим индексы в датафрейме DF. если бы мы прописали drop = False, то была бы еще одна колонка - старые индексы
    # Разделение данных на признаки и целевую переменную
    
    
    # Предварительная обработка категориальных данных
    # Порядковое кодирование. Обучение, трансформация и упаковка в df
    
    ordinal = OrdinalEncoder()
    ordinal.fit(df[cat_columns]);
    Ordinal_encoded = ordinal.transform(df[cat_columns])
    df_ordinal = pd.DataFrame(Ordinal_encoded, columns=cat_columns)
    df[cat_columns] = df_ordinal[cat_columns]
    return df

def scale_frame(frame):
    df = frame.copy()
    X,y = df.drop(columns = ['HeartDisease']), df['HeartDisease']
    scaler = StandardScaler()
    X_scale = scaler.fit_transform(X.values)
    return X_scale, y, scaler

In [3]:
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
df_proc = preprocessing_data_frame(df)
X, y, scaler = scale_frame(df_proc)
# разбиваем на тестовую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y,
                                                  test_size=0.3,
                                                  random_state=42)

In [5]:
def eval_metrics(actual, pred):
    accuracy = accuracy_score(actual, pred)
    precision = precision_score(actual, pred)
    recall = recall_score(actual, pred)
    f1 = f1_score(actual, pred)
    return accuracy, precision, recall, f1

In [7]:
params = {
    'alpha': [0.0008, 0.0009, 0.001, 0.002],
    'l1_ratio': [0.1, 0.3, 0.5, 0.8, 0.9],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'loss': ['log_loss', 'hinge'],
    'fit_intercept': [True, False],
    'eta0': [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'learning_rate': ['optimal', 'invscaling', 'adaptive']
}  
with mlflow.start_run():
    lr = SGDClassifier(random_state=None, class_weight='balanced')
    clf = RandomizedSearchCV(lr, params, cv=3, n_jobs=4, n_iter=10, random_state=None, scoring='recall')
    clf.fit(X_train, y_train)
    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    (accuracy, precision, recall, f1)  = eval_metrics(y_val, y_pred)
    alpha = best.alpha
    l1_ratio = best.l1_ratio
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("l1_ratio", best.l1_ratio)
    mlflow.log_param("penalty", best.penalty)
    mlflow.log_param("loss", best.loss)
    mlflow.log_param("fit_intercept", best.fit_intercept)
    mlflow.log_param("eta0", best.eta0)
    mlflow.log_param("learning_rate", best.learning_rate)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    
    predictions = best.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(best, "model", signature=signature)

2026/04/15 07:28:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 07:28:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gregarious-gull-593 at: http://127.0.0.1:5000/#/experiments/0/runs/3a1aacd5fdbc475fb6fca484ea318fc5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


In [8]:
best.predict(X_val[122:123])[0]

np.int64(1)

In [9]:
y_val.iloc[122]

np.int64(1)

In [10]:
!ls ./mlartifacts/0/

models


In [118]:
## История запуска моделей

In [11]:
dfruns = mlflow.search_runs()
frame = dfruns.sort_values("metrics.recall", ascending=False)
frame

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.accuracy,metrics.f1,metrics.recall,metrics.precision,...,params.fit_intercept,params.l1_ratio,params.learning_rate,params.alpha,params.eta0,params.penalty,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name
0,3a1aacd5fdbc475fb6fca484ea318fc5,0,FINISHED,mlflow-artifacts:/0/3a1aacd5fdbc475fb6fca484ea...,2026-04-15 12:28:28.251000+00:00,2026-04-15 12:28:31.831000+00:00,0.857143,0.859649,0.859649,0.859649,...,True,0.8,optimal,0.0009,0.0001,elasticnet,alina,NOTEBOOK,gregarious-gull-593,mlflow.ipynb
3,22a03cd1baae4092afd0c5b13c55bfb3,0,FINISHED,mlflow-artifacts:/0/22a03cd1baae4092afd0c5b13c...,2026-04-15 08:25:15.186000+00:00,2026-04-15 08:25:17.430000+00:00,0.861607,0.863436,0.859649,0.867257,...,False,0.3,adaptive,0.001,0.1,elasticnet,alina,NOTEBOOK,wise-dolphin-896,mlflow.ipynb
5,7e78056e43df42509c343d12b83b6f5c,0,FINISHED,mlflow-artifacts:/0/7e78056e43df42509c343d12b8...,2026-04-15 08:21:46.922000+00:00,2026-04-15 08:21:58.232000+00:00,0.843750,0.843049,0.824561,0.862385,...,True,0.1,optimal,0.002,0.01,elasticnet,alina,NOTEBOOK,bittersweet-grub-550,mlflow.ipynb
1,891c342af93e4a1196832c7f23248018,0,FINISHED,mlflow-artifacts:/0/891c342af93e4a1196832c7f23...,2026-04-15 12:25:10.443000+00:00,2026-04-15 12:25:17.791000+00:00,0.839286,0.836364,0.807018,0.867925,...,False,0.5,adaptive,0.0009,0.001,l2,alina,NOTEBOOK,indecisive-asp-599,mlflow.ipynb
2,bd4bfd99c30a4e2dbf3175bf23f201d6,0,FINISHED,mlflow-artifacts:/0/bd4bfd99c30a4e2dbf3175bf23...,2026-04-15 08:29:14.859000+00:00,2026-04-15 08:29:17.195000+00:00,0.821429,0.821429,0.807018,0.836364,...,True,0.3,optimal,0.002,0.0001,elasticnet,alina,NOTEBOOK,unique-goat-837,mlflow.ipynb
4,eb569c1814984745b53ab42224947db7,0,FINISHED,mlflow-artifacts:/0/eb569c1814984745b53ab42224...,2026-04-15 08:24:35.297000+00:00,2026-04-15 08:24:39.085000+00:00,0.839286,0.836364,0.807018,0.867925,...,False,0.9,adaptive,0.02,0.001,l2,alina,NOTEBOOK,polite-moth-807,mlflow.ipynb
6,020a0f181c1f4271bff03da199e495df,0,FINISHED,mlflow-artifacts:/0/020a0f181c1f4271bff03da199...,2026-04-08 12:36:11.383000+00:00,2026-04-08 12:36:40.991000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,1e-05,elasticnet,alina,NOTEBOOK,skittish-cow-417,mlflow.ipynb
7,cbfe4e52c4e74347bc7eb74289ba8878,0,FINISHED,mlflow-artifacts:/0/cbfe4e52c4e74347bc7eb74289...,2026-04-08 11:16:11.626000+00:00,2026-04-08 11:16:30.887000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,auspicious-goat-212,mlflow.ipynb
8,5dcd664b9ccf417f88cff16ad1dbdd50,0,FAILED,mlflow-artifacts:/0/5dcd664b9ccf417f88cff16ad1...,2026-04-08 11:13:04.040000+00:00,2026-04-08 11:13:24.794000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,respected-carp-186,mlflow.ipynb
9,a37a396305ac40b195a03c9457c19cc5,0,FAILED,mlflow-artifacts:/0/a37a396305ac40b195a03c9457...,2026-04-08 11:09:24.979000+00:00,2026-04-08 11:09:47.049000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,resilient-bird-578,mlflow.ipynb


In [12]:
dfruns

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.accuracy,metrics.f1,metrics.recall,metrics.precision,...,params.fit_intercept,params.l1_ratio,params.learning_rate,params.alpha,params.eta0,params.penalty,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name
0,3a1aacd5fdbc475fb6fca484ea318fc5,0,FINISHED,mlflow-artifacts:/0/3a1aacd5fdbc475fb6fca484ea...,2026-04-15 12:28:28.251000+00:00,2026-04-15 12:28:31.831000+00:00,0.857143,0.859649,0.859649,0.859649,...,True,0.8,optimal,0.0009,0.0001,elasticnet,alina,NOTEBOOK,gregarious-gull-593,mlflow.ipynb
1,891c342af93e4a1196832c7f23248018,0,FINISHED,mlflow-artifacts:/0/891c342af93e4a1196832c7f23...,2026-04-15 12:25:10.443000+00:00,2026-04-15 12:25:17.791000+00:00,0.839286,0.836364,0.807018,0.867925,...,False,0.5,adaptive,0.0009,0.001,l2,alina,NOTEBOOK,indecisive-asp-599,mlflow.ipynb
2,bd4bfd99c30a4e2dbf3175bf23f201d6,0,FINISHED,mlflow-artifacts:/0/bd4bfd99c30a4e2dbf3175bf23...,2026-04-15 08:29:14.859000+00:00,2026-04-15 08:29:17.195000+00:00,0.821429,0.821429,0.807018,0.836364,...,True,0.3,optimal,0.002,0.0001,elasticnet,alina,NOTEBOOK,unique-goat-837,mlflow.ipynb
3,22a03cd1baae4092afd0c5b13c55bfb3,0,FINISHED,mlflow-artifacts:/0/22a03cd1baae4092afd0c5b13c...,2026-04-15 08:25:15.186000+00:00,2026-04-15 08:25:17.430000+00:00,0.861607,0.863436,0.859649,0.867257,...,False,0.3,adaptive,0.001,0.1,elasticnet,alina,NOTEBOOK,wise-dolphin-896,mlflow.ipynb
4,eb569c1814984745b53ab42224947db7,0,FINISHED,mlflow-artifacts:/0/eb569c1814984745b53ab42224...,2026-04-15 08:24:35.297000+00:00,2026-04-15 08:24:39.085000+00:00,0.839286,0.836364,0.807018,0.867925,...,False,0.9,adaptive,0.02,0.001,l2,alina,NOTEBOOK,polite-moth-807,mlflow.ipynb
5,7e78056e43df42509c343d12b83b6f5c,0,FINISHED,mlflow-artifacts:/0/7e78056e43df42509c343d12b8...,2026-04-15 08:21:46.922000+00:00,2026-04-15 08:21:58.232000+00:00,0.843750,0.843049,0.824561,0.862385,...,True,0.1,optimal,0.002,0.01,elasticnet,alina,NOTEBOOK,bittersweet-grub-550,mlflow.ipynb
6,020a0f181c1f4271bff03da199e495df,0,FINISHED,mlflow-artifacts:/0/020a0f181c1f4271bff03da199...,2026-04-08 12:36:11.383000+00:00,2026-04-08 12:36:40.991000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,1e-05,elasticnet,alina,NOTEBOOK,skittish-cow-417,mlflow.ipynb
7,cbfe4e52c4e74347bc7eb74289ba8878,0,FINISHED,mlflow-artifacts:/0/cbfe4e52c4e74347bc7eb74289...,2026-04-08 11:16:11.626000+00:00,2026-04-08 11:16:30.887000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,auspicious-goat-212,mlflow.ipynb
8,5dcd664b9ccf417f88cff16ad1dbdd50,0,FAILED,mlflow-artifacts:/0/5dcd664b9ccf417f88cff16ad1...,2026-04-08 11:13:04.040000+00:00,2026-04-08 11:13:24.794000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,respected-carp-186,mlflow.ipynb
9,a37a396305ac40b195a03c9457c19cc5,0,FAILED,mlflow-artifacts:/0/a37a396305ac40b195a03c9457...,2026-04-08 11:09:24.979000+00:00,2026-04-08 11:09:47.049000+00:00,0.843750,0.838710,0.798246,0.883495,...,True,0.9,optimal,0.001,0.0001,elasticnet,alina,NOTEBOOK,resilient-bird-578,mlflow.ipynb


In [18]:
path2model = '/home/alina/lab_3/mlartifacts/0/models/m-3df51cbfc19b414ca8765a61db07f44e/artifacts/'
print(path2model)

/home/alina/lab_3/mlartifacts/0/models/m-3df51cbfc19b414ca8765a61db07f44e/artifacts/


In [19]:
loaded_model = mlflow.sklearn.load_model(path2model)
loaded_model

,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'elasticnet'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0009
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.8
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"n_jobs n_jobs: int, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [22]:
test_patient = pd.DataFrame([{
    'Age': 50,
    'Sex': 'M',
    'ChestPainType': 'ASY',
    'RestingBP': 130,
    'Cholesterol': 200,
    'FastingBS': 0,
    'RestingECG': 'Normal',
    'MaxHR': 150,
    'ExerciseAngina': 'Y',
    'Oldpeak': 1.5,
    'ST_Slope': 'Flat',
    'HeartDisease': 0
}])

test_input = scaler.transform(preprocessing_data_frame(test_patient).drop(columns=['HeartDisease']))
prediction = loaded_model.predict(test_input)
prediction

/home/alina/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


array([1])

### Запускает модель в сервис локально 

In [26]:
!mlflow models serve -m $path2model -p 5001 --env-manager local

2026/04/15 10:33:43 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/04/15 10:33:43 INFO mlflow.pyfunc.backend: === Running command 'exec uvicorn --host 127.0.0.1 --port 5001 --workers 1 mlflow.pyfunc.scoring_server.app:app'
INFO:     Started server process [202153]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)
INFO:     127.0.0.1:55806 - "POST /invocations HTTP/1.1" 200 OK
INFO:     127.0.0.1:41772 - "POST /invocations HTTP/1.1" 200 OK
INFO:     Shutting down
^C

Aborted!
INFO:     Finished server process [202153]
ERROR:    Traceback (most recent call last):
  File "/home/alina/miniconda3/lib/python3.13/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/home/alina/miniconda3/lib/python3.13/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
      

### Тестовый запрос к модели через curl

In [ ]:
!curl http://127.0.0.1:5001/invocations -H"Content-Type:application/json"  --data '{"inputs": [[0.5, 1.0, 1.2, 0.1, -0.5, 0.0, 1.0, -0.8, 1.0, 0.5, -1.0]]}'

mlflow.ipynb  mlruns
